# 07 · Final models and hidden-set predictions

One "best" model per dataset, chosen by the experiments in notebooks 02–05:
* **Kryptonite-20/30/40:** sparse polynomial logistic regression (greedy selection + degree-10 polynomial on the
  selected bits, $C$ chosen on validation data).  Selection and $C$ use an 80/20 split of all labelled data; the
  final model is refitted on all labelled data.
* **Kryptonite-40-Adapt:** KryptoNet-40 with LoRA on $W_1$, rank 1, trained on five different 80/20 splits of the
  4,000 labelled examples (early stopping on the 20%); the hidden-set prediction is the majority vote of the five.

Sanity checks (no hidden labels are available): agreement with an independently trained MLP, with the analytic
rank-1 swap of KryptoNet-40, the predicted class balance, and `check_submission.py`.

In [1]:
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import numpy as np

from krypto.data import load, load_hidden, holdout, ROOT, NS
from krypto.polynomial import SparsePolyLogReg
from krypto.selection import greedy_forward_selection
from krypto.mlp import mlp, train, logits, set_seed
from krypto.lora import load_base, add_lora, swap_oracle
from krypto.stats import load_json, mean_ci, save_json
from krypto.energy import track

OUT = ROOT / "hiddenlabels"
OUT.mkdir(exist_ok=True)
mlp_config = {int(k): v for k, v in load_json("mlp_config.json").items()}
expected = load_json("model_comparison.json")
summary = {}

In [2]:
for n in NS:
    X, y = load(f"kryptonite-{n}")
    H = load_hidden(f"kryptonite-{n}")
    with track(f"07_final_n{n}"):
        X_fit, y_fit, X_b, y_b = holdout(X, y, seed=0)                # 80% fit / 20% model selection
        S, _ = greedy_forward_selection(X_fit, y_fit, X_b, y_b)
        C = max((0.85, 10, 100, 1000, 10000), key=lambda C: SparsePolyLogReg(S, 10, C).fit(X_fit, y_fit).score(X_b, y_b))
        final = SparsePolyLogReg(S, 10, C).fit(X, y)                 # refit on all labelled data
        pred = final.predict(H).astype(np.int64)
        set_seed(0)
        net, _ = train(mlp(n, 512, **mlp_config[n]), X_fit, y_fit, X_b, y_b, seed=0)
        agree_mlp = float(((logits(net, H) > 0).astype(int) == pred).mean())
    np.save(OUT / f"y_predicted_{n}.npy", pred)
    mu, h = mean_ci(expected["accuracy"][str(n)]["Sparse polynomial LR (ours)"]) if expected else (np.nan, np.nan)
    summary[str(n)] = {"S": sorted(S), "C": C, "train_acc": final.score(X, y), "agreement_with_MLP": agree_mlp,
                       "positive_rate": float(pred.mean()), "expected_accuracy": [mu, h]}
    print(f"n={n}: S={sorted(S)}, C={C}, training accuracy {final.score(X, y):.4f}, hidden P(y=1)={pred.mean():.3f}, "
          f"agreement with MLP {agree_mlp:.4f}; expected accuracy (5 splits) {100 * mu:.1f} ± {100 * h:.1f}%")

[track] 07_final_n20: 56.2 s


n=20: S=[0, 1, 2, 4, 5, 7, 9, 10, 12, 15], C=100, training accuracy 0.9594, hidden P(y=1)=0.498, agreement with MLP 0.9771; expected accuracy (5 splits) 95.6 ± 0.9%


[track] 07_final_n30: 138.2 s


n=30: S=[2, 5, 10, 11, 14, 17, 19, 20, 26, 28], C=100, training accuracy 0.9614, hidden P(y=1)=0.501, agreement with MLP 0.9929; expected accuracy (5 splits) 96.0 ± 0.4%


[track] 07_final_n40: 153.5 s


n=40: S=[6, 7, 12, 15, 21, 24, 25, 30, 36, 39], C=10, training accuracy 0.9602, hidden P(y=1)=0.504, agreement with MLP 0.9945; expected accuracy (5 splits) 95.8 ± 0.2%


## Kryptonite-40-Adapt with LoRA (rank 1 on the first layer)

In [3]:
X, y = load("kryptonite-40-adapt")
H = load_hidden("kryptonite-40-adapt")
base = load_base()
lora_lrs = load_json("kryptonet_methods.json")["lr"]["LoRA, W1, r=1"]
LR = max(set(lora_lrs), key=lora_lrs.count)                         # learning rate most often chosen in notebook 05
print("learning rate:", LR)
votes, val_accs = [], []
with track("07_final_adapt"):
    for seed in range(5):
        X_tr, y_tr, X_va, y_va = holdout(X, y, seed)                 # 80% train, 20% for early stopping
        set_seed(seed)
        net, info = train(add_lora(base, 1, (0,)), X_tr, y_tr, X_va, y_va, lr=LR, batch_size=64, seed=seed)
        votes.append((logits(net, H) > 0).astype(int))
        val_accs.append(info["val_acc"])
pred = (np.mean(votes, 0) > 0.5).astype(np.int64)
oracle = (logits(swap_oracle(base, 0, 19), H) > 0).astype(int)
np.save(OUT / "y_predicted_40_adapt.npy", pred)
summary["40_adapt"] = {"val_accs": val_accs, "agreement_with_rank1_swap": float((pred == oracle).mean()),
                       "pairwise_agreement_of_members": float(np.mean([(a == b).mean() for i, a in enumerate(votes)
                                                                      for b in votes[i + 1:]])),
                       "positive_rate": float(pred.mean())}
print(f"LoRA ensemble: member validation accuracies {np.round(val_accs, 4).tolist()}; "
      f"agreement with the analytic rank-1 swap on the hidden set {summary['40_adapt']['agreement_with_rank1_swap']:.4f}; "
      f"hidden P(y=1)={pred.mean():.3f}")
save_json(summary, "hidden_predictions_summary.json")

learning rate: 0.0003


[track] 07_final_adapt: 45.7 s
LoRA ensemble: member validation accuracies [0.96, 0.9575, 0.9588, 0.965, 0.9512]; agreement with the analytic rank-1 swap on the hidden set 0.9977; hidden P(y=1)=0.512


## Format check with the coursework's `check_submission.py`

In [4]:
print(subprocess.run([sys.executable, str(ROOT / "check_submission.py"), str(ROOT)], capture_output=True, text=True).stdout)

OK    y_predicted_20.npy
OK    y_predicted_30.npy
OK    y_predicted_40.npy
OK    y_predicted_40_adapt.npy
All files are in the right format.

